# 1. Data Exploration - PASCAL VOC 2012

## Muc tieu
- Download va tim hieu dataset PASCAL VOC 2012
- Phan tich phan bo classes, kich thuoc anh, so luong objects
- Hieu ro du lieu truoc khi train model

## Dataset Overview
- **PASCAL VOC 2012**: 20 object classes
- **Train**: ~5,717 images | **Val**: ~5,823 images
- **Annotation**: XML format (bounding boxes + class labels)
- **Classes**: aeroplane, bicycle, bird, boat, bottle, bus, car, cat, chair, cow, diningtable, dog, horse, motorbike, person, pottedplant, sheep, sofa, train, tvmonitor

In [ ]:
import sys
sys.path.append("../src")

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from PIL import Image
from tqdm import tqdm

from dataset import (
    VOC_CLASSES, CLASS_TO_IDX, IDX_TO_CLASS,
    parse_voc_annotation, download_voc_dataset, get_voc_datasets
)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

%matplotlib inline

## 1.1 Download Dataset

In [ ]:
DATA_ROOT = "../data"
YEAR = "2012"

voc_root = os.path.join(DATA_ROOT, f"VOCdevkit/VOC{YEAR}")

if not os.path.exists(voc_root):
    voc_root = download_voc_dataset(DATA_ROOT, YEAR)
    print(f"Dataset downloaded to: {voc_root}")
else:
    print(f"Dataset already exists at: {voc_root}")

# Kiem tra cau truc thu muc
for d in ["JPEGImages", "Annotations", "ImageSets/Main"]:
    path = os.path.join(voc_root, d)
    if os.path.exists(path):
        count = len(os.listdir(path))
        print(f"  {d}: {count} files")
    else:
        print(f"  {d}: NOT FOUND")

## 1.2 Phan tich phan bo classes

Dem so luong objects cua tung class trong tap train va val de hieu su can bang cua dataset.

In [ ]:
def analyze_split(voc_root, split_name):
    """Phan tich mot split (train/val) cua VOC dataset."""
    split_file = os.path.join(voc_root, "ImageSets", "Main", f"{split_name}.txt")
    with open(split_file, "r") as f:
        image_ids = [line.strip() for line in f.readlines() if line.strip()]

    class_counts = Counter()
    objects_per_image = []
    image_sizes = []
    box_sizes = []  # (width, height) cua moi bounding box
    box_areas = []
    aspect_ratios = []

    annotation_dir = os.path.join(voc_root, "Annotations")
    image_dir = os.path.join(voc_root, "JPEGImages")

    for img_id in tqdm(image_ids, desc=f"Analyzing {split_name}"):
        # Parse annotation
        ann_path = os.path.join(annotation_dir, f"{img_id}.xml")
        ann = parse_voc_annotation(ann_path)

        # Dem classes
        for label in ann["labels"]:
            class_name = IDX_TO_CLASS[label]
            class_counts[class_name] += 1

        objects_per_image.append(len(ann["labels"]))

        # Kich thuoc anh
        img_path = os.path.join(image_dir, f"{img_id}.jpg")
        img = Image.open(img_path)
        image_sizes.append(img.size)  # (width, height)

        # Kich thuoc bounding boxes
        for box in ann["boxes"]:
            w = box[2] - box[0]
            h = box[3] - box[1]
            box_sizes.append((w, h))
            box_areas.append(w * h)
            if h > 0:
                aspect_ratios.append(w / h)

    return {
        "num_images": len(image_ids),
        "class_counts": class_counts,
        "objects_per_image": objects_per_image,
        "image_sizes": image_sizes,
        "box_sizes": box_sizes,
        "box_areas": box_areas,
        "aspect_ratios": aspect_ratios,
    }

# Phan tich train va val
train_stats = analyze_split(voc_root, "train")
val_stats = analyze_split(voc_root, "val")

print(f"Train: {train_stats['num_images']} images, {sum(train_stats['class_counts'].values())} objects")
print(f"Val:   {val_stats['num_images']} images, {sum(val_stats['class_counts'].values())} objects")

In [ ]:
# ---- Bieu do 1: Phan bo classes (train + val) ----
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, stats, title in zip(axes, [train_stats, val_stats], ["Train", "Val"]):
    classes = VOC_CLASSES
    counts = [stats["class_counts"].get(c, 0) for c in classes]
    sorted_idx = np.argsort(counts)[::-1]
    sorted_classes = [classes[i] for i in sorted_idx]
    sorted_counts = [counts[i] for i in sorted_idx]

    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(classes)))
    ax.barh(range(len(classes)), sorted_counts, color=colors)
    ax.set_yticks(range(len(classes)))
    ax.set_yticklabels(sorted_classes)
    ax.set_xlabel("Number of instances")
    ax.set_title(f"{title} Set - Class Distribution")
    ax.invert_yaxis()

    for i, count in enumerate(sorted_counts):
        ax.text(count + 10, i, str(count), va="center", fontsize=9)

plt.tight_layout()
plt.savefig("../results/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

# Nhan xet
print("\nNhan xet:")
most_common = train_stats["class_counts"].most_common(3)
least_common = train_stats["class_counts"].most_common()[-3:]
print(f"  Classes nhieu nhat: {', '.join([f'{c}({n})' for c, n in most_common])}")
print(f"  Classes it nhat:    {', '.join([f'{c}({n})' for c, n in least_common])}")
print(f"  => Dataset KHONG can bang - class 'person' chiem da so")

## 1.3 Phan tich so objects per image va kich thuoc bounding boxes

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. So objects per image
ax = axes[0, 0]
ax.hist(train_stats["objects_per_image"], bins=range(1, max(train_stats["objects_per_image"]) + 2),
        edgecolor="black", alpha=0.7, color="#2196F3")
ax.set_xlabel("Number of objects per image")
ax.set_ylabel("Frequency")
ax.set_title("Objects per Image Distribution (Train)")
ax.axvline(np.mean(train_stats["objects_per_image"]), color="red", linestyle="--",
           label=f"Mean: {np.mean(train_stats['objects_per_image']):.1f}")
ax.legend()

# 2. Phan bo kich thuoc anh
ax = axes[0, 1]
widths = [s[0] for s in train_stats["image_sizes"]]
heights = [s[1] for s in train_stats["image_sizes"]]
ax.scatter(widths, heights, alpha=0.3, s=10, c="#FF9800")
ax.set_xlabel("Width (px)")
ax.set_ylabel("Height (px)")
ax.set_title("Image Size Distribution (Train)")
ax.axhline(np.mean(heights), color="red", linestyle="--", alpha=0.5)
ax.axvline(np.mean(widths), color="red", linestyle="--", alpha=0.5)

# 3. Phan bo dien tich bounding box
ax = axes[1, 0]
areas = np.array(train_stats["box_areas"])
# Phan loai theo COCO: small (<32^2), medium (32^2-96^2), large (>96^2)
small = np.sum(areas < 32**2)
medium = np.sum((areas >= 32**2) & (areas < 96**2))
large = np.sum(areas >= 96**2)
ax.bar(["Small\n(<32x32)", "Medium\n(32-96)", "Large\n(>96x96)"],
       [small, medium, large], color=["#f44336", "#ff9800", "#4caf50"])
ax.set_ylabel("Number of boxes")
ax.set_title("Bounding Box Size Distribution")
for i, v in enumerate([small, medium, large]):
    ax.text(i, v + 50, str(v), ha="center", fontsize=11)

# 4. Aspect ratio
ax = axes[1, 1]
ratios = np.array(train_stats["aspect_ratios"])
ratios_clipped = np.clip(ratios, 0, 5)
ax.hist(ratios_clipped, bins=50, edgecolor="black", alpha=0.7, color="#9C27B0")
ax.set_xlabel("Aspect Ratio (width/height)")
ax.set_ylabel("Frequency")
ax.set_title("Bounding Box Aspect Ratio Distribution")
ax.axvline(1.0, color="red", linestyle="--", label="Square (1:1)")
ax.legend()

plt.tight_layout()
plt.savefig("../results/data_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nThong ke:")
print(f"  Trung binh objects/image: {np.mean(train_stats['objects_per_image']):.1f}")
print(f"  Kich thuoc anh trung binh: {np.mean(widths):.0f} x {np.mean(heights):.0f}")
print(f"  Box area trung binh: {np.mean(areas):.0f} px^2")
print(f"  Small/Medium/Large: {small}/{medium}/{large}")

## 1.4 Hien thi mot so anh mau voi bounding boxes

In [ ]:
from visualize import draw_boxes
from dataset import get_voc_datasets

train_dataset, val_dataset = get_voc_datasets(DATA_ROOT, YEAR)

# Hien thi 6 anh mau
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
np.random.seed(42)
sample_indices = np.random.choice(len(train_dataset), 6, replace=False)

for ax, idx in zip(axes.flat, sample_indices):
    image, target = train_dataset[idx]
    # Convert tensor to numpy (C,H,W) -> (H,W,C)
    img_np = image.permute(1, 2, 0).numpy()
    ax.imshow(img_np)

    boxes = target["boxes"].numpy()
    labels = target["labels"].numpy()

    for box, label in zip(boxes, labels):
        x1, y1, x2, y2 = box
        class_name = IDX_TO_CLASS[label]
        import matplotlib.patches as patches
        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor="lime", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 3, class_name, fontsize=8, color="white",
                bbox=dict(boxstyle="round,pad=0.2", facecolor="green", alpha=0.7))

    ax.set_title(f"Image idx={idx}, {len(labels)} objects")
    ax.axis("off")

plt.suptitle("Sample Images with Ground Truth Bounding Boxes", fontsize=14)
plt.tight_layout()
plt.savefig("../results/sample_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 1.5 Ket luan Data Exploration

**Nhung dieu can luu y khi train model:**

1. **Class imbalance**: Class "person" chiem da so -> model co the bias ve person. Can theo doi per-class AP.
2. **Da dang kich thuoc**: Co nhieu small objects -> two-stage detectors (Faster R-CNN) thuong xu ly tot hon. YOLO can multi-scale detection.
3. **Aspect ratios da dang**: Boxes co nhieu ty le khac nhau -> anchor-based models can anchor ratios phu hop, anchor-free (YOLOv8) co loi the tu dong.
4. **So objects/image**: Trung binh ~2-3 objects/image, nhung co anh len toi 10+ -> NMS threshold quan trong.

**Thong tin nay se duoc su dung de phan tich ket qua cua tung model o cac notebooks tiep theo.**